> **Experiment E03: Leak-free per-turbine client dataset**
> Spec: [E03](../docs/experiments/E03_client_dataset.md) · Config: [clients.yaml](../configs/clients.yaml)
> Registered in `experiments/registry.yaml`.

# Build the shared client partitions
One turbine is one client. Arm D and federated training load the same saved arrays. This CPU notebook does not train a model or import TensorFlow.

On Kaggle attach the CARE data and `dist/windturbine-client-bridge.zip` as datasets, then Run All. Internet is optional. For internet mode set `WT_REPO_REF` to a published commit containing the bridge. A notebook uploaded alone cannot import unpublished code. Output stays in `/kaggle/working/windturbine_clients`. Save that output as a Kaggle Dataset for the training notebook.

Locally run from the checkout and set `WT_DATA_ROOT`. Optional `WT_CLIENT_OUTPUT_ROOT` selects a new output directory. Existing output is never recursively deleted. Set `WT_CLIENT_SMOKE=1` only for the small synthetic check; those outputs are not research results.

## 1. Find the shared source and a writable output folder

In [1]:
import os, sys, json, zipfile, subprocess, hashlib
from pathlib import Path
from dataclasses import replace
import numpy as np
import pandas as pd

SMOKE = os.environ.get("WT_CLIENT_SMOKE", "0") == "1"
WORK_ROOT = Path(os.environ.get("WT_WORK_ROOT", "/kaggle/working" if Path("/kaggle/input").exists() else str(Path.cwd())))
WORK_ROOT.mkdir(parents=True, exist_ok=True)
MODULE = Path("src/windturbine/data/clients.py")
explicit_root, requested_ref = os.environ.get("WT_REPO_ROOT"), os.environ.get("WT_REPO_REF")
if explicit_root and requested_ref:
    raise ValueError("Choose WT_REPO_ROOT or WT_REPO_REF, not both")
REPO, bundle_sha256, resolved_ref = None, None, None
if explicit_root:
    REPO = Path(explicit_root)
    if not (REPO / MODULE).is_file():
        raise RuntimeError("WT_REPO_ROOT does not contain the client bridge: " + str(REPO))
elif requested_ref:
    REPO = WORK_ROOT / "windturbine-client-bridge-git"
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/PranavMundhra/Wind-Turbine.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", requested_ref], check=True)
    resolved_ref = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], check=True, capture_output=True, text=True).stdout.strip()
    expected_ref = subprocess.run(["git", "-C", str(REPO), "rev-parse", requested_ref + "^{commit}"], check=True, capture_output=True, text=True).stdout.strip()
    assert resolved_ref == expected_ref, "Requested source revision was not selected"
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    if Path("/kaggle/input").exists():
        candidates += [p.parent.parent.parent.parent for p in Path("/kaggle/input").rglob("src/windturbine/data/clients.py")]
    REPO = next((p for p in candidates if (p / MODULE).is_file()), None)
    if REPO is None and Path("/kaggle/input").exists():
        bundles = list(Path("/kaggle/input").rglob("windturbine-client-bridge.zip"))
        if len(bundles) > 1:
            raise RuntimeError("Attach only one source ZIP, or set WT_REPO_ROOT")
        if bundles:
            bundle_sha256 = hashlib.sha256(bundles[0].read_bytes()).hexdigest()
            target = WORK_ROOT / ("windturbine-source-" + bundle_sha256[:16])
            with zipfile.ZipFile(bundles[0]) as archive:
                names = archive.namelist()
                if len(names) != len(set(names)):
                    raise ValueError("Duplicate paths in source archive")
                for name in names:
                    if "\\" in name or not (target / name).resolve().is_relative_to(target.resolve()):
                        raise ValueError("Unsafe source archive path")
                package_manifest = json.loads(archive.read("bridge_bundle_manifest.json"))
                if package_manifest.get("format_version") != 1 or not isinstance(package_manifest.get("files"), dict):
                    raise ValueError("Unsupported source bundle manifest")
                files = package_manifest["files"]
                if set(names) != set(files) | {"bridge_bundle_manifest.json"}:
                    raise ValueError("Source bundle file list does not match manifest")
                for name, digest in files.items():
                    if hashlib.sha256(archive.read(name)).hexdigest() != digest:
                        raise ValueError("Source bundle hash mismatch: " + name)
                archive.extractall(target)
            REPO = target
if REPO is None or not (REPO / MODULE).is_file():
    raise RuntimeError("Attach windturbine-client-bridge.zip as a Kaggle Dataset (internet off works), "
                       "or attach a source checkout and set WT_REPO_ROOT. Optional internet mode: "
                       "set WT_REPO_REF to a published commit/tag containing src/windturbine/data/clients.py.")
# Also verify an already extracted source bundle before importing any package code.
if (REPO / "bridge_bundle_manifest.json").is_file():
    package_manifest = json.loads((REPO / "bridge_bundle_manifest.json").read_text(encoding="utf-8"))
    if package_manifest.get("format_version") != 1:
        raise ValueError("Unsupported extracted source bundle")
    for name, digest in package_manifest["files"].items():
        source_path = (REPO / name).resolve()
        if not source_path.is_relative_to(REPO.resolve()) or hashlib.sha256(source_path.read_bytes()).hexdigest() != digest:
            raise ValueError("Extracted source bundle hash mismatch: " + name)
source_notebooks = [REPO / "notebooks" / name for name in ("13_arm_local_turbine.ipynb", "20_federated_clients.ipynb")]
source_files = sorted([*REPO.glob("src/windturbine/**/*.py"), *REPO.glob("configs/*.yaml"),
                       *(p for p in source_notebooks if p.is_file())])
source_hashes = {p.relative_to(REPO).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest() for p in source_files}
SOURCE_IDENTITY = {"source_tree_sha256": hashlib.sha256(json.dumps(source_hashes, sort_keys=True).encode()).hexdigest(),
                   "source_files": source_hashes, "bundle_sha256": bundle_sha256,
                   "requested_git_ref": requested_ref, "resolved_git_commit": resolved_ref}
sys.path.insert(0, str(REPO / "src"))
from windturbine.config import load_config
from windturbine.data.clients import build_client_dataset, load_client_dataset
OUTPUT_ROOT = Path(os.environ.get("WT_CLIENT_OUTPUT_ROOT", os.environ.get("WT_CLIENT_OUTPUT", str(WORK_ROOT / "windturbine_clients"))))
if Path("/kaggle/input").exists() and not OUTPUT_ROOT.resolve().is_relative_to(Path("/kaggle/working").resolve()):
    raise ValueError("On Kaggle, WT_CLIENT_OUTPUT_ROOT must be inside /kaggle/working")
print("Source:", REPO, "| Output:", OUTPUT_ROOT)
print("Source identity:", json.dumps(SOURCE_IDENTITY, indent=2))

Source: C:\Users\pooja\Desktop\wind turbine\Wind-Turbine | Output: D:\wt_work\windturbine_clients
Source identity: {
  "source_tree_sha256": "c8f83a45376374b2fcf30e7a85c165ff4526d1b275deb48b7064091495a5fe0a",
  "source_files": {
    "configs/arm_isolated.yaml": "ab352789a22a1d3cfe52d6854ebe095a04e85be89b10e0d51beedd3694ec8af1",
    "configs/arm_local_turbine.yaml": "8db53eded3df967c36e65c4a4d3712438a24581bc2a3abb6fb74520a59261795",
    "configs/arm_pooled.yaml": "c2c70de648928f873760115a5f405f3abb44ca6fe663cb9ee7054cb79c69cfb7",
    "configs/arm_pooled_dedup.yaml": "485e5470df67831de28dde349d7874554857f9cf9b4d0b7ccba1ba717f40b34d",
    "configs/base.yaml": "16668bbdc1a309e8c60d5c8fcc7d831bed2b7fb5b72eb28cfbb19d461a6e7533",
    "configs/clients.yaml": "41cdfee241855f0eb93d228d471b9a8c1224214155b539a25e96e5f0b1121f52",
    "configs/fed_fedavg.yaml": "bbf4cc2e43fe406561836d9b0d07ffa6a50b133962251f3b10f782e37424ccb5",
    "configs/fed_fedprox.yaml": "a6e4d3aaef05ab6b3214ceb635deceea6cf1b98

## 2. Load the actual experiment configuration
Settings come from this checkout's YAML files. The full-data default requires all 22 turbines and 58 events. A small synthetic fixture only checks execution and data contracts.

In [2]:
CFG = load_config(REPO / "configs/base.yaml", REPO / "configs/clients.yaml")
CFG = replace(CFG, seed=int(os.environ.get("WT_SEED", CFG.seed)))
EXPECTED_CLIENTS = int(CFG.extra.get("expected_clients", 22))
EXPECTED_EVENTS = int(CFG.extra.get("expected_events", 58))
if SMOKE:
    DATA_ROOT = WORK_ROOT / "synthetic_care"
    farm_dir = DATA_ROOT / CFG.farm
    datasets = farm_dir / "datasets"
    datasets.mkdir(parents=True, exist_ok=True)
    metadata = []
    for eid in range(1, 5):
        aid = (eid - 1) // 2 + 1
        times = pd.date_range("2024-01-01", periods=100, freq="10min")
        values = np.arange(100, dtype=float) + eid * 1000
        frame = pd.DataFrame({"time_stamp": times, "asset_id": aid,
                              "train_test": ["train"] * 80 + ["prediction"] * 20,
                              "status_type_id": 0, "sensor_0_avg": values,
                              "wind_speed_3_avg": 6.0 + values / 10000,
                              "power_30_avg": 0.5 + values / 10000})
        # Deliberately repeat one training row in this turbine's prediction period.
        frame.loc[80, ["sensor_0_avg", "wind_speed_3_avg", "power_30_avg"]] = frame.loc[0, ["sensor_0_avg", "wind_speed_3_avg", "power_30_avg"]].to_numpy()
        frame.to_csv(datasets / f"comma_{eid}.csv", index=False)
        metadata.append({"event_id": eid, "event_label": "anomaly" if eid % 2 else "normal",
                         "event_start": times[85], "event_end": times[99]})
    pd.DataFrame(metadata).to_csv(farm_dir / "comma_event_info.csv", index=False)
    pd.DataFrame({"sensor_name": ["sensor_0", "wind_speed_3", "power_30"],
                  "description": ["temperature", "wind speed", "active power"]}).to_csv(farm_dir / "comma_feature_description.csv", index=False)
    CFG = replace(CFG, val_block_hours=1, val_split_mode="blocked")
    EXPECTED_CLIENTS, EXPECTED_EVENTS = 2, 4
else:
    from windturbine.paths import resolve_data_root
    DATA_ROOT = Path(resolve_data_root(cfg=CFG))
print("Data:", DATA_ROOT)
print("Config:", CFG)

Data: D:\care_data
Config: Config(farm='Wind Farm C', dataset_root='data/raw', cache_dir='data/cache', use_avg_only=True, drop_reactive=False, angle_transform=True, max_nan_fraction=0.3, normal_status_ids=(0, 2), wind_speed_cutin=3.0, wind_speed_rated=12.0, power_near_zero_thresh=0.02, hidden_layers=(133, 83, 20, 83, 133), learning_rate=0.0056, use_lr_plateau=True, lr_plateau_factor=0.5, lr_plateau_patience=4, noise_std=0.0, batch_size=64, max_epochs=200, early_stop_patience=10, seed=42, val_fraction=0.15, val_split_mode='blocked', val_block_hours=120, score_quantile=0.99, criticality_threshold=72, beta=0.5, care_weights=(1, 1, 1, 2), arm='clients', output_dir='results/runs', extra={'client_unit': 'asset_id', 'remove_rows_seen_in_prediction': True, 'fingerprint_decimals': 6, 'expected_clients': 22, 'expected_events': 58})


## 3. Build, clean and save
The shared code selects normal training rows, applies the configured per-event split, removes matching fingerprints from training/validation, and fits imputation and scaling on retained training rows only. Every client has the same ordered feature names. A client's inactive features are recorded explicitly.

Fingerprint checks compare selected sensor values rounded to the configured precision; they are not proof that all temporal or statistical dependence has disappeared. The split audit states when a short event needs chronological fallback. `blocked` means blocks of retained rows, not guaranteed calendar intervals. No large raw-data copy or ZIP is created. Old parsed caches without source provenance are not used.

In [3]:
manifest = build_client_dataset(DATA_ROOT, OUTPUT_ROOT, CFG, cache_dir=None,
                               expected_clients=EXPECTED_CLIENTS, expected_events=EXPECTED_EVENTS,
                               source_identity=SOURCE_IDENTITY)
assert len(manifest["clients"]) == EXPECTED_CLIENTS
assert manifest["n_events"] == EXPECTED_EVENTS
print("Saved manifest:", OUTPUT_ROOT / "dataset_manifest.json")
print("Schema hash:", manifest.get("schema_hash"))
print("Preprocessing hash:", manifest.get("preprocessing_hash"))
print("Row removal and overlap audit:")
print(json.dumps(manifest["audit"], indent=2))

Saved manifest: D:\wt_work\windturbine_clients\dataset_manifest.json
Schema hash: c27ebf3195b641eb5df74438219e69a34e0b64787ab3e1e5c7e1220c24e22034
Preprocessing hash: 602c93e4a97d99cb24fb2013e34e72a8399d568053e98b5abba05edd5606cd52
Row removal and overlap audit:
{
  "2": {
    "train_rows_before": 81614,
    "val_rows_before": 12960,
    "train_removed_prediction": 280,
    "val_removed_prediction": 0,
    "train_removed_validation": 11520,
    "intersections": {
      "train_val": 0,
      "train_prediction": 0,
      "val_prediction": 0,
      "train_global_prediction": 0,
      "val_global_prediction": 0,
      "train_global_validation": 0
    },
    "chronological_fallback_events": [],
    "train_rows": 69814,
    "val_rows": 12960,
    "prediction_rows": 6435,
    "train_unique_fingerprints": 38839,
    "val_unique_fingerprints": 12960,
    "prediction_unique_fingerprints": 6377
  },
  "5": {
    "train_rows_before": 40382,
    "val_rows_before": 6480,
    "train_removed_predictio

## 4. Reload every client and check the training contract
Reloading verifies saved artifact hashes. Then check all three fingerprint intersections independently. Prediction rows include event IDs, timestamps and a status-only evaluation mask. The extra power-curve rule applies to training, not evaluation.

In [4]:
reference_schema = None
client_summary = []
for aid in manifest["clients"]:
    client = load_client_dataset(OUTPUT_ROOT, aid, verify=True)
    if reference_schema is None:
        reference_schema = client["schema"]
    assert client["schema"] == reference_schema, "Ordered features differ between clients"
    fingerprints = {s: set(client[s]["fingerprint"].tolist()) for s in ("train", "val", "prediction")}
    assert not fingerprints["train"] & fingerprints["val"]
    assert not fingerprints["train"] & fingerprints["prediction"]
    assert not fingerprints["val"] & fingerprints["prediction"]
    for split in ("train", "val", "prediction"):
        batch = client[split]
        assert batch["X"].shape[1] == len(reference_schema)
        assert np.isfinite(batch["X"]).all()
        assert len(batch["X"]) == len(batch["event_id"]) == len(batch["time_stamp"])
    assert len(client["prediction"]["eval_mask"]) == len(client["prediction"]["X"])
    client_summary.append({"asset_id": aid, **{s: len(client[s]["X"]) for s in ("train", "val", "prediction")}})
summary = pd.DataFrame(client_summary)
print(summary.to_string(index=False))
print("Verified clients:", len(summary), "| Shared input width:", len(reference_schema))
# This is the exact matrix interface a future FL client will use.
first_client = load_client_dataset(OUTPUT_ROOT, manifest["clients"][0], verify=True)
X_batch = first_client["train"]["X"][:CFG.batch_size]
assert X_batch.ndim == 2 and len(X_batch) > 0
print("Example training batch:", X_batch.shape, X_batch.dtype)

 asset_id  train   val  prediction
        2  69814 12960        6435
        5  40382  6480        2737
       12  91451 18296        8001
       13  34560  5369        2305
       14  38204  5760        1585
       15  81199 12814        2459
       16 155468 24480       10203
       21  67967 12685        2609
       23 108068 18720        8280
       32  65960 12753        3404
       33  73837 12960        5864
       34 137880 25427       10650
       35 121987 23582       15568
       38  62912 11520        3677
       42  65639 12240        6497
       43  91899 22650       13672
       44  71596 12240       14044
       50  71153 12240        5435
       52 101905 21313       10143
       53 117628 31620       11536
       55 109173 18500        8799
       56  66355 12960        4625
Verified clients: 22 | Shared input width: 243
Example training batch: (64, 243) float32


## 5. Next step: rerun the local baseline before comparing FL
Save this output as a Kaggle Dataset. In `13_arm_local_turbine.ipynb`, set `WT_CLIENT_DATASET` to its folder containing `dataset_manifest.json`. Arm D loads these exact processed arrays, with no second scaler or cleaning path. Rerun D: its old scores used a different preprocessing path and cannot establish a matched baseline.

A first FedAvg implementation should exchange model weights in this exact feature order, keep client preprocessing local, weight client updates by retained training rows, and use the same validation threshold and evaluation mask as D. Passing these checks enables implementation; it does not demonstrate an FL accuracy gain. Full-data execution and matched saved runs are still required.